# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'Nebula project page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/LocateAnything-3B
Updated
5 days ago
•
35.8k
•
739
openbmb/MiniCPM5-1B
Updated
7 days ago
•
45.7k
•
678
LiquidAI/LFM2.5-8B-A1B
Updated
about 22 hours ago
•
37.9k
•
360
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
Apr 17
•
2.53M
•
1.21k
meituan-longcat/LongCat-Video-Av

In [21]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n5 days ago\n•\n35.8k\n•\n739\nopenbmb/MiniCPM5-1B\nUpdated\n7 days ago\n•\n45.7k\n•\n678\nL

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and collaboration platform at the forefront of the machine learning (ML) revolution. Serving as a central hub for ML researchers, engineers, developers, and enthusiasts, Hugging Face empowers the next generation of innovators to build open, ethical AI solutions together.

With a fast-growing global community, Hugging Face hosts **over 2 million machine learning models**, **500,000 datasets**, and countless applications — all accessible in one place. The platform facilitates seamless collaboration to create, discover, and deploy ML models and datasets, accelerating innovation across industries.

---

## Our Mission

“To build an open and ethical AI future by enabling collaboration and transparency in machine learning.”

Hugging Face believes in democratizing access to AI technology, fostering a community-driven ecosystem where knowledge and resources are freely shared to advance the field responsibly.

---

## What We Offer

### The Hugging Face Hub  
- A centralized platform to upload, share, explore, and experiment with state-of-the-art ML models and datasets.  
- Hosting for unlimited public repositories, encouraging open-source contribution and collective growth.

### Models  
- Browse millions of pre-trained models across NLP, computer vision, audio, reinforcement learning, and more.  
- Popular models updated regularly by the community and teams such as NVIDIA, OpenBMB, and others.

### Datasets  
- Access to a vast collection of datasets to train and evaluate models, including structured Wikipedia data, language corpora, and curated domain-specific sets.

### Spaces  
- Interactive ML applications and demos hosted directly in your browser for hands-on testing and community engagement.

### Enterprise Solutions  
- Tailored support, inference endpoints, and scalable infrastructure for businesses integrating AI into their products.  
- Hugging Face PRO for enterprise-grade collaboration and deployment.

---

## Our Community & Culture

At the heart of Hugging Face is a passionate, diverse, and welcoming community dedicated to open science and ethical AI development. The culture promotes:

- **Collaboration:** Encouraging sharing of models, code, datasets, and ideas via forums, Discord, GitHub, and community blogs.  
- **Innovation:** Pioneering research and engineering pushing the boundaries of natural language processing (NLP), computer vision, audio processing, and other AI fields.  
- **Inclusivity & Ethics:** Building AI that is fair, transparent, and beneficial to all users and society.

Join thousands of contributors and users from academia, industry, and hobbyist backgrounds who make Hugging Face a dynamic learning environment and innovation hub.

---

## Careers at Hugging Face

Hugging Face is continuously growing and looking for talented, passionate individuals to join their teams in roles such as:

- Machine Learning Engineers & Researchers  
- Software Engineers & DevOps  
- Data Scientists  
- Community Managers & Developer Advocates  
- Product Managers  

Employees thrive in an open, fast-paced startup atmosphere focused on learning, collaboration, and pushing technological boundaries. Working here means contributing to projects used worldwide by millions and shaping the future of AI.

Interested candidates can find more details and apply via the **Careers** section on the Hugging Face website.

---

## Customers & Partners

Hugging Face serves a wide range of users and organizations, including:

- AI researchers and academic institutions advancing ML science.  
- Technology companies integrating open-source models into their products.  
- Enterprises seeking customizable, scalable AI solutions with expert support.  
- Individual developers and hobbyists expanding the capabilities of machine learning.

Notable community contributors include NVIDIA, IBM Research, and numerous AI labs and startups collaborating on open benchmarks, datasets, and model improvement.

---

## Connect With Us

- Website: [huggingface.co](https://huggingface.co)  
- GitHub: github.com/huggingface  
- Discord & Forum: Engage with the community, ask questions, share ideas  
- Blog: Stay updated with case studies, research breakthroughs, and tutorials  
- Social: Twitter, LinkedIn for latest news and events

---

*Hugging Face — The AI community building the future.*  
Join the movement shaping open, collaborative AI for all.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community shaping the future of machine learning. It serves as a comprehensive collaboration platform where developers, researchers, and organizations come together to create, share, and deploy state-of-the-art machine learning models, datasets, and applications.

With over 2 million models and 500,000+ datasets available, Hugging Face empowers the machine learning ecosystem to innovate faster and smarter.

---

## What We Offer

- **Models and Datasets:** Access and contribute to a vast library of over 2 million pre-trained models and 500k datasets, covering diverse AI tasks such as natural language processing, computer vision, audio analysis, and more.
  
- **Spaces:** Host and explore cutting-edge AI applications like image generation, object detection, and video avatar creation seamlessly in-browser.

- **Buckets & Storage:** Secure storage infrastructure for hosting and managing machine learning assets.

- **Enterprise Solutions:** Tailored support, inference endpoints, and professional services to help businesses scale AI deployment efficiently.

- **HuggingChat:** An advanced chat interface offering interactive AI applications.

---

## Community & Collaboration

At its core, Hugging Face is a collaborative platform fueling innovation through open-source contributions and community engagement.

- **Community Forums and Discord:** Connect with thousands of AI enthusiasts, share insights, and get support.
  
- **Blogs & Daily Papers:** Stay updated with research articles, tutorials, and the latest breakthroughs in AI.

- **GitHub Repositories:** Participate in open-source projects, contribute code, and explore AI frameworks.

- **Events and Partnerships:** Engage through workshops, hackathons, and strategic industry partnerships.

---

## Customers

Hugging Face caters to a broad spectrum of users including:

- Academic researchers pioneering new AI techniques.
- Startups and large enterprises deploying AI-powered solutions.
- Developers building AI applications across industries such as healthcare, finance, creative media, and robotics.

Its scalable infrastructure supports both individual users and enterprise teams with advanced AI capabilities.

---

## Company Culture

Hugging Face fosters an open, inclusive, and innovation-driven culture. The company values community, transparency, and ethical AI development. By encouraging collaboration and knowledge sharing, it nurtures an environment where creativity and scientific rigor thrive side by side.

---

## Careers at Hugging Face

Join a passionate team dedicated to building the future of AI!

- Opportunities in software engineering, machine learning research, product management, and community engagement.
- Work with cutting-edge AI technologies and impact millions of users worldwide.
- Be part of a mission-driven company that emphasizes open source, continuous learning, and a supportive work environment.

Explore job openings and grow your career in one of the most exciting fields today.

---

## Get Involved

- Browse 2M+ models and 500k+ datasets at [huggingface.co](https://huggingface.co)
- Join the conversation on [Discord](https://discord.gg/huggingface)
- Explore docs and tutorials to start building AI applications.
- Participate in community challenges and contribute to open source projects.

---

**Hugging Face — The AI Community Building the Future**

Empowering the world's machine learning community to collaborate and innovate better, faster, and together.

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face: Your AI Buddy for the Future 🤗🤖

---

## Who We Are

Welcome to **Hugging Face**, where the AI community doesn't just build models—they build the future with a whole lot of collaboration, code, and of course, heart! Think of us as the ultimate hangout spot for machine learning enthusiasts: researchers, developers, enterprises, and curious cats alike.

---

## What’s the Buzz About?

- 🌟 **2 Million+ Models**  
  Models galore! From Nvidia’s *LocateAnything* (it sees all) to the chatty *HuggingChat*, we've got it all.

- 📚 **500,000+ Datasets**  
  Feeding your AI's appetite with the biggest and best data smorgasbord on the web.

- 💻 **Spaces & Apps**  
  Run state-of-the-art image generation in your browser or check out talking heads via *LongCat-Video-Avatar*. Your AI playground awaits!

- 🚀 **Enterprise & PRO Support**  
  Serious AI business? We offer scalable solutions and professional support so your enterprise can flex its AI muscles without breaking a sweat.

---

## The Hugging Face Ecosystem: Where Collaboration Meets Innovation

- **Models, Datasets, and Apps** live and breathe here—updated frequently by a buzzing community (think thousands of contributors and users).
- **HuggingChat**: Our own chatty AI ready to talk your ears off (or help automate your customer support).
- **Spaces**: Think mini AI theme parks running in real time inside your browser.
- **Buckets & Endpoints**: Store your data, infer like a pro, and keep your ML workflows smooth and jazzy.

---

## Culture: More Hugs, Less Bugs

At Hugging Face, we’re not just about cutting-edge AI—we’re about cutting-edge **community spirit**. Here’s why people love working and collaborating with us:

- **Open & Collaborative:** Share your work, get feedback, and build on others' ideas without gatekeepers.
- **Passion-driven:** Our community is fueled by curiosity and the thrill of AI breakthroughs.
- **Diversity of Thought:** We welcome all walks of AI-life—from academic wizards to startup hustlers.
- **Playful Seriousness:** Yes, we work hard, but we laugh harder. Expect memes, emojis, and a friendly code of conduct.

---

## Careers: Join the AI Party 🎉

If you’re wild about AI and want to be part of a community that’s **building the future together**, Hugging Face wants YOU!

- Roles for Machine Learning Engineers, Research Scientists, DevOps wizards, and Community Buddies.
- Work remotely or from hubs where the coffee flows as freely as the open-source code.
- Grow your skills by collaborating with some of the brightest minds in AI.
- Perks? Competitive salaries and a team that celebrates the fusion of tech + heart.

---

## For Customers & Investors: Why Hugging Face?

- **Proven Tech Powerhouse:** Serving millions of developers and enterprises worldwide.
- **Open-source Magic:** Our models and tools are embraced by the top AI companies and universities globally.
- **Scalable Solutions:** Enterprise-ready infrastructure tailored to your business needs.
- **Community Innovation Engine:** Always fresh, always moving forward, driven by tens of thousands of active contributors.

---

## Still Curious?

Dive in, explore [Hugging Face](https://huggingface.co), and let AI be your new best friend.

Join the conversation on **Discord**, explore endless models & datasets, or launch your own AI Space. The future is collaborative and cuddly!

---

### Hugging Face: Where algorithms meet hugs — come for the AI, stay for the community! 🤗

---

*Brand Colors: A sunny #FFD21E, a vibrant #FF9D00, and a trusty #6B7280—just like us, warm, energetic, and dependable.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>